---
title: CONUS 404 diagnostic plots
author: Harsha R. Hampapura
date: 09-02-2026
---

### Data Access

- This notebook illustrates how to make diagnostic plots using the CONUS 404 dataset hosted on NCAR's Geoscience Data Exchange (GDEX).
- https://gdex.ucar.edu/datasets/d559000/
- This data is open access and can be accessed via 3 protocols
  1) POSIX (if you have access to NCAR's HPC systems: Casper or Derecho)
  2) HTTPS
  3) OSDF using intake-ESM catalogs.
- Learn about intake-ESM catalogs: https://intake-esm.readthedocs.io/en/stable/ 

In [2]:
# Imports 
import intake
import numpy as np
import pandas as pd
import xarray as xr
import seaborn as sns
import matplotlib.pyplot as plt
import os

In [3]:
import dask 
from dask_jobqueue import PBSCluster
from dask.distributed import Client

In [4]:
# Catalog URLs
cat_url     = '/gdex/data/d559000/catalogs/d559000-posix.json' # POSIX access
# cat_url     = 'https://osdf-director.osg-htc.org/ncar/gdex/d559000/catalogs/d559000-osdf.json'
print(cat_url)

/gdex/data/d559000/catalogs/d559000-posix.json


In [5]:
# Set up your scratch folder path
username       = os.environ["USER"]
glade_scratch  = "/glade/derecho/scratch/" + username
print(glade_scratch)

/glade/derecho/scratch/harshah


## Create a PBS cluster

In [6]:
# Create a PBS cluster object
cluster = PBSCluster(
    job_name = 'dask-wk25-hpc',
    cores = 1,
    memory = '10GiB',
    processes = 1,
    local_directory = glade_scratch+'/dask/spill/',
    log_directory = glade_scratch + '/dask/logs/',
    resource_spec = 'select=1:ncpus=1:mem=10GB',
    queue = 'casper',
    walltime = '5:00:00',
    #interface = 'ib0'
    interface = 'ext'
)

/glade/u/home/harshah/.conda/envs/zarr3/lib/python3.11/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 44747 instead
  warnings.warn(


In [7]:
# Scale the cluster and display cluster dashboard URL
n_workers = 4
client = Client(cluster)
cluster.scale(n_workers)
client.wait_for_workers(n_workers = n_workers)
cluster

PBSCluster(210188f2, 'tcp://128.117.208.101:40763', workers=4, threads=4, memory=40.00 GiB)

## Load CONUS 404 data from GDEX using an intake catalog

In [8]:
col = intake.open_esm_datastore(cat_url)
col

,unique
path,2
variable,206
format,1
short_name,206
long_name,214
units,31
start_time,1
end_time,1
level,1
level_units,1


- col.df turns the catalog object into a pandas dataframe!
- (Actually, it accesses the dataframe attribute of the catalog)

In [9]:
col.df

,path,variable,format,short_name,long_name,units,start_time,end_time,level,level_units,frequency
0,/glade/campaign/collections/gdex/data/d559000/...,ACDEWC,reference,ACDEWC,"Accumulated canopy dew rate, accumulated over ...",mm,1979-10-01,2022-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
1,/glade/campaign/collections/gdex/data/d559000/...,ACDRIPR,reference,ACDRIPR,"Accumulated canopy precipitation drip rate, ac...",mm,1979-10-01,2022-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
2,/glade/campaign/collections/gdex/data/d559000/...,ACDRIPS,reference,ACDRIPS,"Accumulated canopy snow drip rate, accumulated...",mm,1979-10-01,2022-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
3,/glade/campaign/collections/gdex/data/d559000/...,ACECAN,reference,ACECAN,Accumulated net evaporation of canopy water (e...,mm,1979-10-01,2022-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
4,/glade/campaign/collections/gdex/data/d559000/...,ACEDIR,reference,ACEDIR,Accumulated net soil evaporation or snowpack s...,mm,1979-10-01,2022-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
...,...,...,...,...,...,...,...,...,...,...,...
210,/glade/campaign/collections/gdex/data/d559000/...,V,reference,V,V-component of wind with respect to model grid,m s-1,1979-10-01,2022-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
211,/glade/campaign/collections/gdex/data/d559000/...,W,reference,W,W-component of wind,m s-1,1979-10-01,2022-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
212,/glade/campaign/collections/gdex/data/d559000/...,Z,reference,Z,Geopotential height (PH + PHB)/9.81,meters,1979-10-01,2022-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00
213,/glade/campaign/collections/gdex/data/d559000/...,ilev,reference,ilev,vertical stagger levels,Dimensionless,1979-10-01,2022-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00


## Select data and plot

#### What if you don't know the variable names ?
- Use pandas logic to print out the short_name and long_name

In [10]:
col.df[['variable','long_name']]

,variable,long_name
0,ACDEWC,"Accumulated canopy dew rate, accumulated over ..."
1,ACDRIPR,"Accumulated canopy precipitation drip rate, ac..."
2,ACDRIPS,"Accumulated canopy snow drip rate, accumulated..."
3,ACECAN,Accumulated net evaporation of canopy water (e...
4,ACEDIR,Accumulated net soil evaporation or snowpack s...
...,...,...
210,V,V-component of wind with respect to model grid
211,W,W-component of wind
212,Z,Geopotential height (PH + PHB)/9.81
213,ilev,vertical stagger levels


- We notice that long_name is not available for some variables like 'V'
- In such cases, please look at the wrfout_datadictionary file on this page https://gdex.ucar.edu/datasets/d559000/documentation/#

### Temperature
- Plot temperature for a random date

In [11]:
cat_temp = col.search(variable='T2')
cat_temp.df.head()

,path,variable,format,short_name,long_name,units,start_time,end_time,level,level_units,frequency
0,/glade/campaign/collections/gdex/data/d559000/...,T2,reference,T2,TEMP at 2 M,K,1979-10-01,2022-09-30 23:00:00,<NA>,<NA>,0 days 01:00:00


- The data is organized in (virtual) zarr stores with one water year's worth of data in one file
- Select a year. This is done by selcting the start time to be Oct 1 of that year or the end time to be Sep 30 of the same year
- This also means that if you want to request data for other days, say Jan 1 for the year YYYY, you first have to load the data for one year i.e., YYYY and then select the data for that particular day. This example is discussed below.


In [12]:
# date = "2020-10-01"
# # year = "2021"
# cat_temp_subset = cat_temp.search(start_time = date)
cat_temp.df['path'].values

<ArrowExtensionArray>
['/glade/campaign/collections/gdex/data/d559000/kerchunk/wrf2d.parq']
Length: 1, dtype: large_string[pyarrow]

### Load data into xarray

In [1]:
test = xr.open_dataset('/gdex/data/d559000/kerchunk/wy1980.2d.json',engine='kerchunk')
test

NameError: name 'xr' is not defined

In [ ]:
%%time
# Load catalog entries for subset into a dictionary of xarray datasets, and open the first one.
dsets = cat_temp.to_dataset_dict(xarray_open_kwargs={'engine':'kerchunk'})
#
print(f"\nDataset dictionary keys:\n {dsets.keys()}")

In [ ]:
# Load the first dataset and display a summary.
dataset_key = list(dsets.keys())[0]
# store_name = dataset_key + ".zarr"
print(dsets.keys())
ds = dsets[dataset_key]
ds = ds.T2
ds

In [ ]:
%%time
desired_time = "2021-01-01T00"
ds.sel(Time=desired_time,method='nearest').plot(cmap='inferno')

In [ ]:
cluster.close()